In [0]:
dbutils.widgets.text("catalog", "tpch_dev", "Catalog Name")
catalog = dbutils.widgets.get("catalog")

In [0]:
# Gold: dim_geography - One row per nation including parent region
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_geography AS
SELECT
    n.n_nationkey AS geography_key,
    n.n_nationkey AS nation_key,
    n.n_name  AS nation_name,
    n.n_comment    AS nation_comment,
    r.r_regionkey  AS region_key,
    r.r_name AS region_name,
    r.r_comment    AS region_comment
FROM {catalog}.silver.nation n
JOIN {catalog}.silver.region r ON n.n_regionkey = r.r_regionkey
""")
display(spark.table(f"{catalog}.gold.dim_geography"))

In [0]:
# Gold: dim_customer - One row per customer with geography
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_customer AS
SELECT
    c.c_custkey   AS customer_key,
    c.c_name AS customer_name,
    c.c_address   AS customer_address,
    c.c_phone     AS customer_phone,
    c.c_acctbal   AS account_balance,
    c.c_mktsegment AS market_segment,
    c.c_comment   AS customer_comment,
    c.c_nationkey AS geography_key,
    n.n_name AS nation_name,
    r.r_name AS region_name
FROM {catalog}.silver.customers c
LEFT JOIN {catalog}.silver.nation n  ON c.c_nationkey = n.n_nationkey
LEFT JOIN {catalog}.silver.region r ON n.n_regionkey  = r.r_regionkey
""")
display(spark.table(f"{catalog}.gold.dim_customer").limit(10))

In [0]:
# Gold: dim_product - One row per product
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_product AS
SELECT
    p.p_partkey     AS product_key,
    p.p_name  AS product_name,
    p.p_mfgr   AS manufacturer,
    p.p_brand  AS brand,
    p.p_type   AS product_type,
    p.p_size         AS product_size,
    p.p_container AS container,
    p.p_retailprice  AS retail_price,
    p.p_comment      AS product_comment
FROM {catalog}.silver.part p
""")
display(spark.table(f"{catalog}.gold.dim_product").limit(10))

In [0]:
# Gold: dim_supplier - One row per supplier with geography
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_supplier AS
SELECT
    s.s_suppkey   AS supplier_key,
    s.s_name AS supplier_name,
    s.s_address   AS supplier_address,
    s.s_phone     AS supplier_phone,
    s.s_acctbal   AS account_balance,
    s.s_comment   AS supplier_comment,
    s.s_nationkey AS geography_key,
    n.n_name AS nation_name,
    r.r_name AS region_name
FROM {catalog}.silver.supplier s
LEFT JOIN {catalog}.silver.nation n  ON s.s_nationkey = n.n_nationkey
LEFT JOIN {catalog}.silver.region r ON n.n_regionkey  = r.r_regionkey
""")
display(spark.table(f"{catalog}.gold.dim_supplier").limit(10))

In [0]:
# Gold: fact_sales - One row per order line with keys, measures, and attributes
# Distinguishes customer geography from supplier geography
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.fact_sales AS
SELECT
    l.l_orderkey     AS order_key,
    l.l_linenumber   AS line_number,
    o.o_custkey      AS customer_key,
    l.l_partkey      AS product_key,
    l.l_suppkey      AS supplier_key,
    c.c_nationkey    AS customer_geography_key,
    s.s_nationkey    AS supplier_geography_key,
    o.o_orderdate    AS order_date,
    l.l_shipdate     AS ship_date,
    l.l_commitdate   AS commit_date,
    l.l_receiptdate  AS receipt_date,
    o.o_orderstatus  AS order_status,
    l.l_linestatus   AS line_status,
    l.l_returnflag   AS return_flag,
    o.o_orderpriority AS order_priority,
    o.o_clerk        AS order_clerk,
    o.o_shippriority AS ship_priority,
    l.l_shipmode     AS ship_mode,
    l.l_shipinstruct AS ship_instruct,
    l.l_quantity     AS quantity,
    l.l_extendedprice AS extended_price,
    l.l_discount     AS discount_rate,
    l.l_tax          AS tax_rate,
    l.l_extendedprice * l.l_discount                     AS discount_amount,
    l.l_extendedprice * (1 - l.l_discount)               AS revenue,
    l.l_extendedprice * (1 - l.l_discount) * l.l_tax      AS tax_amount,
    l.l_extendedprice * (1 - l.l_discount) * (1 + l.l_tax) AS total_amount_after_tax,
    o.o_totalprice   AS order_total_price,
    l.l_comment      AS line_comment
FROM {catalog}.silver.lineitem l
JOIN {catalog}.silver.orders o    ON l.l_orderkey = o.o_orderkey
LEFT JOIN {catalog}.silver.customers c ON o.o_custkey = c.c_custkey
LEFT JOIN {catalog}.silver.supplier s  ON l.l_suppkey = s.s_suppkey
""")
display(spark.table(f"{catalog}.gold.fact_sales").limit(10))

In [0]:
# Gold: vW_annual_sales_kpis - Annual revenue, orders, active customers, avg order value
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.vW_annual_sales_kpis AS
SELECT
    YEAR(order_date)           AS sales_year,
    SUM(revenue)           AS annual_revenue,
    COUNT(DISTINCT order_key)   AS total_orders,
    COUNT(DISTINCT customer_key) AS active_customers,
    SUM(revenue) / COUNT(DISTINCT order_key) AS avg_order_value
FROM {catalog}.gold.fact_sales
GROUP BY YEAR(order_date)
ORDER BY sales_year
""")
display(spark.table(f"{catalog}.gold.vW_annual_sales_kpis"))

In [0]:
# Reconciliation: Gold revenue and row count vs independent Silver calculation
gold_stats = spark.sql(f"""
SELECT
    COUNT(*)                 AS gold_row_count,
    SUM(revenue)         AS gold_net_revenue,
    SUM(total_amount_after_tax)        AS gold_total_amount
FROM {catalog}.gold.fact_sales
""").collect()[0]

silver_stats = spark.sql(f"""
SELECT
    COUNT(*)                                           AS silver_row_count,
    SUM(l.l_extendedprice * (1 - l.l_discount))         AS silver_net_revenue,
    SUM(l.l_extendedprice * (1 - l.l_discount) * (1 + l.l_tax)) AS silver_total_amount
FROM {catalog}.silver.lineitem l
""").collect()[0]

row_diff   = gold_stats['gold_row_count'] - silver_stats['silver_row_count']
rev_diff   = float(gold_stats['gold_net_revenue']) - float(silver_stats['silver_net_revenue'])
total_diff = float(gold_stats['gold_total_amount']) - float(silver_stats['silver_total_amount'])

print("=" * 70)
print("GOLD vs SILVER RECONCILIATION REPORT")
print("=" * 70)
print(f"{'Metric':<25} {'Gold':>20} {'Silver':>20}")
print("-" * 70)
print(f"{'Row Count':<25} {gold_stats['gold_row_count']:>20,} {silver_stats['silver_row_count']:>20,}")
print(f"{'Net Revenue':<25} {float(gold_stats['gold_net_revenue']):>20,.2f} {float(silver_stats['silver_net_revenue']):>20,.2f}")
print(f"{'Total Amount':<25} {float(gold_stats['gold_total_amount']):>20,.2f} {float(silver_stats['silver_total_amount']):>20,.2f}")
print("-" * 70)
print(f"{'Row Count Diff':<25} {row_diff:>20,}")
print(f"{'Net Revenue Diff':<25} {rev_diff:>20,.2f}")
print(f"{'Total Amount Diff':<25} {total_diff:>20,.2f}")
print("-" * 70)

passed = (row_diff == 0 and abs(rev_diff) < 0.01 and abs(total_diff) < 0.01)
status = "✅ PASS" if passed else "❌ FAIL"
print(f"Reconciliation Status: {status}")
print("=" * 70)

In [0]:
# Validate dimension key uniqueness in each dimension table
dims = [
    ("dim_geography", "geography_key"),
    ("dim_customer",  "customer_key"),
    ("dim_product",   "product_key"),
    ("dim_supplier",  "supplier_key"),
]

print("=" * 65)
print("DIMENSION KEY UNIQUENESS VALIDATION")
print("=" * 65)
print(f"{'Table':<20} {'Key':<20} {'Rows':>10} {'Distinct':>10} {'Status':>8}")
print("-" * 65)

all_unique = True
for table, key in dims:
    df = spark.table(f"{catalog}.gold.{table}")
    total   = df.count()
    distinct = df.select(key).distinct().count()
    is_unique = (total == distinct)
    if not is_unique:
        all_unique = False
    status = "✅ PASS" if is_unique else "❌ FAIL"
    print(f"{table:<20} {key:<20} {total:>10,} {distinct:>10,} {status:>8}")

print("-" * 65)
print(f"Overall Uniqueness: {'✅ PASS' if all_unique else '❌ FAIL'}")
print("=" * 65)